In [13]:
import pandas as pd
import os

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 50)

# Load the parquet file
g_z_all = pd.read_parquet('../historical g spread/bond_z.parquet')

# Now display info - it will show all columns
g_z_all.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 431056 entries, 0 to 431055
Data columns (total 44 columns):
 #   Column                     Non-Null Count   Dtype   
---  ------                     --------------   -----   
 0   Security_1                 431056 non-null  object  
 1   Security_2                 431056 non-null  object  
 2   Last_Spread                431056 non-null  float64 
 3   Z_Score                    431056 non-null  float64 
 4   Max                        431056 non-null  float64 
 5   Min                        431056 non-null  float64 
 6   Last_vs_Max                431056 non-null  float64 
 7   Last_vs_Min                431056 non-null  float64 
 8   Percentile                 431056 non-null  float64 
 9   XCCY                       431056 non-null  float64 
 10  Custom_Sector_1            431056 non-null  object  
 11  Custom_Sector_2            431056 non-null  object  
 12  Rating_1                   431056 non-null  object  
 13  Rating_2      

## All Canadian Bonds

In [14]:
# All Canadian Bonds
g_z_cad = g_z_all[
    (g_z_all['Currency_1'] == 'CAD') & 
    (g_z_all['Currency_2'] == 'CAD')
]
# Select columns
g_z_select_columns = g_z_all[[
    'Security_1',
    'Security_2', 
    'Last_Spread',
    'Z_Score',
    'Max',
    'Min',
    'Last_vs_Max',
    'Last_vs_Min',
    'Percentile',
    'Custom_Sector_1',
    'Custom_Sector_2',
    'Yrs_Mat_Bucket_1',
    'Yrs_Mat_Bucket_2',
    'Size @ Best Offer_runs1',
    'Size @ Best Bid_runs2',
    'Bid/Offer_runs1',
    'Bid/Offer_runs2'
]].copy()

# Sort by Z_Score descending
g_z_select_columns = g_z_select_columns.sort_values(by='Z_Score', ascending=False)


# Identify float columns except the two size columns
size_cols = ['Size @ Best Offer_runs1', 'Size @ Best Bid_runs2']
float_cols = [col for col in g_z_select_columns.columns if g_z_select_columns[col].dtype == 'float' and col not in size_cols]

# Build the format dictionary
format_dict = {col: '{:.1f}' for col in float_cols}
format_dict.update({col: '{:,.0f}' for col in size_cols})

# Display with custom formatting
display(
    g_z_select_columns.head(50).style.format(format_dict)
)

,Security_1,Security_2,Last_Spread,Z_Score,Max,Min,Last_vs_Max,Last_vs_Min,Percentile,Custom_Sector_1,Custom_Sector_2,Yrs_Mat_Bucket_1,Yrs_Mat_Bucket_2,Size @ Best Offer_runs1,Size @ Best Bid_runs2,Bid/Offer_runs1,Bid/Offer_runs2
0,NSANY 1.85 09/16/26,NVACN 7 7/8 07/23/26,656.1,7.4,656.1,-336.7,0.0,992.8,100.0,Auto,HY,1-2.1,1-2.1,nan,"2,000,000",nan,nan
1,F 6.8 05/12/28,NVACN 7 7/8 07/23/26,579.6,7.4,579.6,-353.0,0.0,932.6,100.0,Auto,HY,2.1-3.1,1-2.1,nan,"2,000,000",nan,nan
2,IAGCN 6.921 09/30/2084,NVACN 7 7/8 07/23/26,725.3,7.4,725.3,-196.2,0.0,921.5,100.0,Financial Hybrid,HY,>25.1,1-2.1,"2,000,000","2,000,000",13.0,nan
3,MFCCN 7.117 06/19/2082,NVACN 7 7/8 07/23/26,633.9,7.3,633.9,-296.4,0.0,930.3,100.0,Financial Hybrid,HY,>25.1,1-2.1,nan,"2,000,000",nan,nan
4,CM 6.987 07/28/2084,NVACN 7 7/8 07/23/26,725.9,7.3,725.9,-197.7,0.0,923.6,100.0,Financial Hybrid,HY,>25.1,1-2.1,"2,000,000","2,000,000",18.0,nan
5,IFCCN 7.338 06/30/2083,NVACN 7 7/8 07/23/26,656.8,7.3,656.8,-260.4,0.0,917.2,100.0,Financial Hybrid,HY,>25.1,1-2.1,nan,"2,000,000",nan,nan
6,F 5.8 03/08/29,NVACN 7 7/8 07/23/26,604.6,7.3,604.6,-329.5,0.0,934.2,100.0,Auto,HY,3.1-4.1,1-2.1,nan,"2,000,000",nan,nan
7,F 5.85 05/17/27,NVACN 7 7/8 07/23/26,559.5,7.3,559.5,-368.8,0.0,928.2,100.0,Auto,HY,1-2.1,1-2.1,nan,"2,000,000",nan,nan
8,BPYUCN 4 09/30/26,NVACN 7 7/8 07/23/26,620.6,7.3,620.6,-304.9,0.0,925.6,100.0,HY,HY,1-2.1,1-2.1,"2,000,000","2,000,000",73.8,nan
9,BRPCN 5 1/8 06/15/29,NVACN 7 7/8 07/23/26,810.1,7.3,810.1,-106.8,0.0,916.9,100.0,Other,HY,3.1-4.1,1-2.1,nan,"2,000,000",nan,nan


## Same Sector

In [15]:
# Same Sector

# Filter for rows where Custom_Sector_1 equals Custom_Sector_2
g_z_sector = g_z_select_columns[g_z_select_columns['Custom_Sector_1'] == g_z_select_columns['Custom_Sector_2']].copy()

# Identify float columns except the two size columns
size_cols = ['Size @ Best Offer_runs1', 'Size @ Best Bid_runs2']
float_cols = [col for col in g_z_select_columns.columns if g_z_select_columns[col].dtype == 'float' and col not in size_cols]

# Build the format dictionary
format_dict = {col: '{:.1f}' for col in float_cols}
format_dict.update({col: '{:,.0f}' for col in size_cols})

# Display with custom formatting
display(
    g_z_sector.head(30).style.format(format_dict)
)

,Security_1,Security_2,Last_Spread,Z_Score,Max,Min,Last_vs_Max,Last_vs_Min,Percentile,Custom_Sector_1,Custom_Sector_2,Yrs_Mat_Bucket_1,Yrs_Mat_Bucket_2,Size @ Best Offer_runs1,Size @ Best Bid_runs2,Bid/Offer_runs1,Bid/Offer_runs2
8,BPYUCN 4 09/30/26,NVACN 7 7/8 07/23/26,620.6,7.3,620.6,-304.9,0.0,925.6,100.0,HY,HY,1-2.1,1-2.1,"2,000,000","2,000,000",73.8,nan
13,ACACN 4 5/8 08/15/29,NVACN 7 7/8 07/23/26,598.4,7.2,598.4,-324.6,0.0,923.0,100.0,HY,HY,3.1-4.1,1-2.1,"2,000,000","2,000,000",17.0,nan
14,COEQRE 7.45 07/04/29,NVACN 7 7/8 07/23/26,863.1,7.2,863.1,-57.1,0.0,920.2,100.0,HY,HY,3.1-4.1,1-2.1,nan,"2,000,000",nan,nan
21,BPYUCN 7 1/8 02/13/28,NVACN 7 7/8 07/23/26,663.4,7.1,663.4,-264.1,0.0,927.5,100.0,HY,HY,2.1-3.1,1-2.1,nan,"2,000,000",nan,nan
279,KRUGCN 5 3/8 04/09/29,NVACN 7 7/8 07/23/26,708.6,6.5,708.6,-212.3,0.0,920.8,100.0,HY,HY,3.1-4.1,1-2.1,nan,"2,000,000",nan,nan
301,CEUCN 6 7/8 05/24/29,NVACN 7 7/8 07/23/26,730.9,6.5,730.9,-190.7,0.0,921.6,100.0,HY,HY,3.1-4.1,1-2.1,nan,"2,000,000",nan,nan
319,MRCCN 9 1/2 09/26/26,NVACN 7 7/8 07/23/26,654.0,6.5,654.0,-265.3,0.0,919.2,100.0,HY,HY,1-2.1,1-2.1,nan,"2,000,000",nan,nan
875,CCACN 6 1/8 02/27/29,NVACN 7 7/8 07/23/26,596.6,6.1,596.6,-325.4,0.0,922.1,100.0,HY,HY,3.1-4.1,1-2.1,"2,000,000","2,000,000",35.0,nan
888,ATRLCN 5.7 03/26/29,NVACN 7 7/8 07/23/26,501.7,6.0,501.7,-435.3,0.0,937.0,100.0,HY,HY,3.1-4.1,1-2.1,nan,"2,000,000",nan,nan
922,RY 4.612 07/26/27,TD 4.68 01/08/29,-12.3,4.3,-12.3,-21.4,0.0,9.1,100.0,Bail In,Bail In,1-2.1,3.1-4.1,"5,000,000","15,000,000",3.0,3.0


## Same Term

In [16]:
# Same Term

# Filter for rows where Yrs_Mat_Bucket_1 equals Yrs_Mat_Bucket_2
g_z_term = g_z_select_columns[g_z_select_columns['Yrs_Mat_Bucket_1'] == g_z_select_columns['Yrs_Mat_Bucket_2']].copy()

# Identify float columns except the two size columns
size_cols = ['Size @ Best Offer_runs1', 'Size @ Best Bid_runs2']
float_cols = [col for col in g_z_select_columns.columns if g_z_select_columns[col].dtype == 'float' and col not in size_cols]

# Build the format dictionary
format_dict = {col: '{:.1f}' for col in float_cols}
format_dict.update({col: '{:,.0f}' for col in size_cols})

# Display with custom formatting
display(
    g_z_term.head(30).style.format(format_dict)
)

,Security_1,Security_2,Last_Spread,Z_Score,Max,Min,Last_vs_Max,Last_vs_Min,Percentile,Custom_Sector_1,Custom_Sector_2,Yrs_Mat_Bucket_1,Yrs_Mat_Bucket_2,Size @ Best Offer_runs1,Size @ Best Bid_runs2,Bid/Offer_runs1,Bid/Offer_runs2
0,NSANY 1.85 09/16/26,NVACN 7 7/8 07/23/26,656.1,7.4,656.1,-336.7,0.0,992.8,100.0,Auto,HY,1-2.1,1-2.1,nan,"2,000,000",nan,nan
7,F 5.85 05/17/27,NVACN 7 7/8 07/23/26,559.5,7.3,559.5,-368.8,0.0,928.2,100.0,Auto,HY,1-2.1,1-2.1,nan,"2,000,000",nan,nan
8,BPYUCN 4 09/30/26,NVACN 7 7/8 07/23/26,620.6,7.3,620.6,-304.9,0.0,925.6,100.0,HY,HY,1-2.1,1-2.1,"2,000,000","2,000,000",73.8,nan
10,F 5.8 03/05/27,NVACN 7 7/8 07/23/26,549.5,7.3,549.5,-383.0,0.0,932.5,100.0,Auto,HY,1-2.1,1-2.1,nan,"2,000,000",nan,nan
43,F 6.326 11/10/26,NVACN 7 7/8 07/23/26,576.8,6.9,576.8,-352.1,0.0,928.9,100.0,Auto,HY,1-2.1,1-2.1,"2,000,000","2,000,000",6.0,nan
44,GM 5 04/09/27,NVACN 7 7/8 07/23/26,484.7,6.9,484.7,-445.7,0.0,930.3,100.0,Auto,HY,1-2.1,1-2.1,nan,"2,000,000",nan,nan
46,F 5.581 02/22/27,NVACN 7 7/8 07/23/26,590.4,6.9,590.4,-337.5,0.0,927.9,100.0,Auto,HY,1-2.1,1-2.1,"5,000,000","2,000,000",3.0,nan
47,F 2.961 09/16/26,NVACN 7 7/8 07/23/26,542.4,6.9,542.4,-378.6,0.0,920.9,100.0,Auto,HY,1-2.1,1-2.1,"2,000,000","2,000,000",4.0,nan
53,GM 5.4 05/08/27,NVACN 7 7/8 07/23/26,478.9,6.9,478.9,-450.4,0.0,929.4,100.0,nan,HY,1-2.1,1-2.1,nan,"2,000,000",nan,nan
55,NSANY 6.95 09/15/26,NVACN 7 7/8 07/23/26,612.7,6.9,612.7,-312.9,0.0,925.6,100.0,Auto,HY,1-2.1,1-2.1,nan,"2,000,000",nan,nan


## Same Ticker

In [17]:
# Same Ticker

# Filter for CAD currency and where Ticker_1 equals Ticker_2
g_z_ticker = g_z_all[
    (g_z_all['Currency_1'] == 'CAD') & 
    (g_z_all['Currency_2'] == 'CAD') &
    (g_z_all['Ticker_1'] == g_z_all['Ticker_2'])
].copy()

# Select the same columns as before
g_z_ticker_select = g_z_ticker[[
    'Security_1',
    'Security_2', 
    'Last_Spread',
    'Z_Score',
    'Max',
    'Min',
    'Last_vs_Max',
    'Last_vs_Min',
    'Percentile',
    'Custom_Sector_1',
    'Custom_Sector_2',
    'Yrs_Mat_Bucket_1',
    'Yrs_Mat_Bucket_2',
    'Size @ Best Offer_runs1',
    'Size @ Best Bid_runs2',
    'Bid/Offer_runs1',
    'Bid/Offer_runs2'
]].copy()

# Sort by Z_Score descending
g_z_ticker_select = g_z_ticker_select.sort_values(by='Z_Score', ascending=False)

# Identify float columns except the two size columns
size_cols = ['Size @ Best Offer_runs1', 'Size @ Best Bid_runs2']
float_cols = [col for col in g_z_ticker_select.columns if g_z_ticker_select[col].dtype == 'float' and col not in size_cols]

# Build the format dictionary
format_dict = {col: '{:.1f}' for col in float_cols}
format_dict.update({col: '{:,.0f}' for col in size_cols})

# Display with custom formatting
display(
    g_z_ticker_select.head(30).style.format(format_dict)
)

,Security_1,Security_2,Last_Spread,Z_Score,Max,Min,Last_vs_Max,Last_vs_Min,Percentile,Custom_Sector_1,Custom_Sector_2,Yrs_Mat_Bucket_1,Yrs_Mat_Bucket_2,Size @ Best Offer_runs1,Size @ Best Bid_runs2,Bid/Offer_runs1,Bid/Offer_runs2
923,RY 4.612 07/26/27,RY 5.228 06/24/30,-11.1,4.2,-11.1,-25.2,0.0,14.1,100.0,Bail In,Bail In,1-2.1,4.1-5.1,"5,000,000","10,000,000",3.0,5.0
1125,RY 4.632 05/01/28,RY 5.228 06/24/30,-6.1,3.5,-5.6,-20.5,-0.5,14.4,99.6,Bail In,Bail In,2.1-3.1,4.1-5.1,"10,000,000","10,000,000",3.0,5.0
1343,BIP 5.71 07/27/30,BIP 5.789 04/25/52,-63.3,3.3,-63.3,-78.8,0.0,15.5,100.0,Infastructure,Infastructure,4.1-5.1,>25.1,"5,000,000","3,000,000",6.0,5.0
1448,RY 4.642 01/17/28,RY 5.228 06/24/30,-11.5,3.3,-10.7,-20.2,-0.7,8.8,97.2,Bail In,Bail In,2.1-3.1,4.1-5.1,"5,000,000","10,000,000",3.0,5.0
2486,ALACN 5.141 03/14/34,ALACN 8.9 11/10/2083,-93.0,2.9,-93.0,-198.5,0.0,105.5,100.0,Midstream,Non Financial Hybrids,7.1-10.1,>25.1,"2,000,000","2,000,000",5.0,16.0
2529,BIP 5.616 11/14/27,BIP 5.789 04/25/52,-94.4,2.9,-94.4,-109.4,0.0,15.0,100.0,Infastructure,Infastructure,2.1-3.1,>25.1,"3,000,000","3,000,000",5.0,5.0
2532,GEICN 5 3/4 07/12/33,GEICN 8.7 07/12/2083,-98.3,2.9,-98.3,-191.6,0.0,93.3,100.0,Midstream,Non Financial Hybrids,7.1-10.1,>25.1,"3,000,000","2,000,000",5.0,17.0
3407,ALACN 5 1/4 01/11/2082,ALACN 8.9 11/10/2083,33.4,2.8,33.4,-27.2,-0.0,60.6,99.6,Non Financial Hybrids,Non Financial Hybrids,>25.1,>25.1,"2,000,000","2,000,000",15.0,16.0
3653,BMO 4.709 12/07/27,BMO 5.039 05/29/28,-1.1,2.8,-1.1,-6.9,-0.0,5.8,99.6,Bail In,Bail In,2.1-3.1,2.1-3.1,"10,000,000","15,000,000",3.0,3.0
3920,ALACN 2.477 11/30/30,ALACN 8.9 11/10/2083,-130.4,2.8,-130.4,-229.0,0.0,98.6,100.0,Midstream,Non Financial Hybrids,5.1-7.1,>25.1,"5,000,000","2,000,000",-2.0,16.0


# What's Rich vs A Specific Bond

In [18]:
# Filter for CAD currency and where Security_1 equals "FTSCN 2.42 07/18/31"
g_z_bondlevel = g_z_all[
    (g_z_all['Currency_1'] == 'CAD') & 
    (g_z_all['Currency_2'] == 'CAD') &
    (g_z_all['Security_1'] == 'FTSCN 2.42 07/18/31')   # change this to flip rich or cheap 
].copy()

# Select the same columns as before
g_z_bondlevel = g_z_bondlevel[[
    'Security_1',
    'Security_2', 
    'Last_Spread',
    'Z_Score',
    'Max',
    'Min',
    'Last_vs_Max',
    'Last_vs_Min',
    'Percentile',
    'Custom_Sector_1',
    'Custom_Sector_2',
    'Yrs_Mat_Bucket_1',
    'Yrs_Mat_Bucket_2',
    'Size @ Best Offer_runs1',
    'Size @ Best Bid_runs2',
    'Bid/Offer_runs1',
    'Bid/Offer_runs2'
]].copy()

# Sort by Z_Score descending
g_z_bondlevel = g_z_bondlevel.sort_values(by='Z_Score', ascending=False)

# Identify float columns except the two size columns
size_cols = ['Size @ Best Offer_runs1', 'Size @ Best Bid_runs2']
float_cols = [col for col in g_z_bondlevel.columns if g_z_bondlevel[col].dtype == 'float' and col not in size_cols]

# Build the format dictionary
format_dict = {col: '{:.1f}' for col in float_cols}
format_dict.update({col: '{:,.0f}' for col in size_cols})

# Display with custom formatting
display(
    g_z_bondlevel.head(30).style.format(format_dict)
)

,Security_1,Security_2,Last_Spread,Z_Score,Max,Min,Last_vs_Max,Last_vs_Min,Percentile,Custom_Sector_1,Custom_Sector_2,Yrs_Mat_Bucket_1,Yrs_Mat_Bucket_2,Size @ Best Offer_runs1,Size @ Best Bid_runs2,Bid/Offer_runs1,Bid/Offer_runs2
810,FTSCN 2.42 07/18/31,NVACN 7 7/8 07/23/26,462.8,6.2,462.8,-466.1,0.0,928.8,100.0,Utility,HY,5.1-7.1,1-2.1,"2,000,000","2,000,000",10.0,nan
2787,FTSCN 2.42 07/18/31,IFCCN 1.928 12/16/30,35.2,2.9,44.2,-9.9,-9.1,45.1,96.8,Utility,Insurance Senior,5.1-7.1,5.1-7.1,"2,000,000","2,000,000",10.0,nan
14220,FTSCN 2.42 07/18/31,HRUCN 2.633 02/19/27,-23.4,2.3,-22.2,-53.0,-1.2,29.7,98.8,Utility,REIT,5.1-7.1,1-2.1,"2,000,000","3,000,000",10.0,nan
22179,FTSCN 2.42 07/18/31,RCICN 5 1/4 04/15/52,-100.4,2.1,-93.8,-126.2,-6.6,25.8,97.6,Utility,Cable/Telco,5.1-7.1,>25.1,"2,000,000","5,000,000",10.0,5.0
22289,FTSCN 2.42 07/18/31,TCN 2.85 11/13/31,-32.5,2.1,-28.5,-48.7,-3.9,16.3,96.4,Utility,Cable/Telco,5.1-7.1,5.1-7.1,"2,000,000","5,000,000",10.0,nan
23607,FTSCN 2.42 07/18/31,GEICN 8.7 07/12/2083,-183.6,2.1,-182.4,-281.2,-1.2,97.6,99.6,Utility,Non Financial Hybrids,5.1-7.1,>25.1,"2,000,000","2,000,000",10.0,17.0
23961,FTSCN 2.42 07/18/31,TD 1.888 03/08/28,31.1,2.1,40.9,7.7,-9.7,23.5,96.8,Utility,Bail In,5.1-7.1,2.1-3.1,"2,000,000","5,000,000",10.0,10.0
28775,FTSCN 2.42 07/18/31,SPBCN 4 1/4 05/18/28,-170.4,2.0,-166.9,-311.5,-3.6,141.1,98.0,Utility,HY,5.1-7.1,2.1-3.1,"2,000,000","3,000,000",10.0,23.9
32207,FTSCN 2.42 07/18/31,GLCRTR 5.588 09/20/26,-34.7,1.9,-24.0,-145.9,-10.7,111.3,85.3,Utility,Asset Backed Subs,5.1-7.1,1-2.1,"2,000,000",nan,10.0,nan
35504,FTSCN 2.42 07/18/31,IPLCN 5.091 11/27/51,-141.4,1.9,-132.6,-174.9,-8.8,33.5,96.8,Utility,Midstream,5.1-7.1,>25.1,"2,000,000","5,000,000",10.0,nan


## VS What We Own

In [19]:
# Filter for CAD currency and where Own? equals 1
g_z_own = g_z_all[
    (g_z_all['Currency_1'] == 'CAD') & 
    (g_z_all['Currency_2'] == 'CAD') &
    (g_z_all['Own_2'] == 1)
].copy()

# Select the same columns as before
g_z_own_select = g_z_own[[
    'Security_1',
    'Security_2', 
    'Last_Spread',
    'Z_Score',
    'Max',
    'Min',
    'Last_vs_Max',
    'Last_vs_Min',
    'Percentile',
    'Custom_Sector_1',
    'Custom_Sector_2',
    'Yrs_Mat_Bucket_1',
    'Yrs_Mat_Bucket_2',
    'Size @ Best Offer_runs1',
    'Size @ Best Bid_runs2',
    'Bid/Offer_runs1',
    'Bid/Offer_runs2'
]].copy()

# Sort by Z_Score descending
g_z_own_select = g_z_own_select.sort_values(by='Z_Score', ascending=False)

# Identify float columns except the two size columns
size_cols = ['Size @ Best Offer_runs1', 'Size @ Best Bid_runs2']
float_cols = [col for col in g_z_own_select.columns if g_z_own_select[col].dtype == 'float' and col not in size_cols]

# Build the format dictionary
format_dict = {col: '{:.1f}' for col in float_cols}
format_dict.update({col: '{:,.0f}' for col in size_cols})

# Display with custom formatting
display(
    g_z_own_select.head(30).style.format(format_dict)
)

,Security_1,Security_2,Last_Spread,Z_Score,Max,Min,Last_vs_Max,Last_vs_Min,Percentile,Custom_Sector_1,Custom_Sector_2,Yrs_Mat_Bucket_1,Yrs_Mat_Bucket_2,Size @ Best Offer_runs1,Size @ Best Bid_runs2,Bid/Offer_runs1,Bid/Offer_runs2
991,DE 4.95 06/14/27,HOMEQU 6.552 10/18/27,-131.1,3.8,-131.1,-164.1,0.0,32.9,100.0,Industrials,Mortgage Provider,1-2.1,2.1-3.1,"5,000,000","3,000,000",5.0,nan
1048,ALCTRA 4.627 06/13/34,TCN 5 3/4 09/08/33,-36.9,3.6,-36.9,-50.1,0.0,13.2,100.0,Utility,Cable/Telco,7.1-10.1,7.1-10.1,"3,000,000","10,000,000",5.0,5.0
1071,AL 5.4 06/01/28,HOMEQU 6.552 10/18/27,-77.1,3.6,-77.1,-115.1,0.0,38.0,100.0,Non Financial Maple,Mortgage Provider,2.1-3.1,2.1-3.1,"2,000,000","3,000,000",3.0,nan
1087,GLCRTR 4.74 09/20/26,HOMEQU 6.552 10/18/27,-126.3,3.6,-126.3,-164.9,0.0,38.6,100.0,Asset Backed,Mortgage Provider,1-2.1,2.1-3.1,"2,000,000","3,000,000",5.0,nan
1102,EAGCCT 4.916 06/17/29,HOMEQU 6.552 10/18/27,-102.8,3.5,-102.8,-134.7,0.0,31.9,100.0,Asset Backed,Mortgage Provider,3.1-4.1,2.1-3.1,"2,000,000","3,000,000",5.0,nan
1118,ATH 5.113 03/07/29,HOMEQU 6.552 10/18/27,-86.2,3.5,-86.2,-117.7,0.0,31.5,100.0,Financial Maples,Mortgage Provider,3.1-4.1,2.1-3.1,"3,000,000","3,000,000",5.0,nan
1123,FORTFD 4.419 12/23/27,HOMEQU 6.552 10/18/27,-116.9,3.5,-116.9,-153.0,0.0,36.1,100.0,Asset Backed,Mortgage Provider,2.1-3.1,2.1-3.1,"2,000,000","3,000,000",5.0,nan
1129,DE 2.81 01/19/29,HOMEQU 6.552 10/18/27,-130.3,3.5,-130.3,-156.0,0.0,25.8,100.0,Industrials,Mortgage Provider,3.1-4.1,2.1-3.1,"2,000,000","3,000,000",10.0,nan
1167,EAGCCT 4.783 07/17/27,HOMEQU 6.552 10/18/27,-116.1,3.5,-116.1,-156.3,0.0,40.1,100.0,Asset Backed,Mortgage Provider,1-2.1,2.1-3.1,"2,000,000","3,000,000",5.0,nan
1169,CAAIAU 3.454 10/07/41,HOMEQU 6.552 10/18/27,-58.9,3.4,-58.9,-98.3,0.0,39.4,100.0,Airport,Mortgage Provider,10.1-25.1,2.1-3.1,"2,000,000","3,000,000",5.0,nan


## VS What We Own Filtered For What We Can Trade

In [20]:
# Filter for CAD currency, owned securities, and additional size/bid-offer filters
g_z_own_executable = g_z_all[
    (g_z_all['Currency_1'] == 'CAD') & 
    (g_z_all['Currency_2'] == 'CAD') &
    (g_z_all['Own_2'] == 1) &
    (g_z_all['Size @ Best Offer_runs1'] > 2000000) &
    (g_z_all['Size @ Best Bid_runs2'] > 2000000) &
    (g_z_all['Bid/Offer_runs1'] < 3) &
    (g_z_all['Bid/Offer_runs2'] < 3)
].copy()

# Select the same columns as before
g_z_own_executable_select = g_z_own_executable[[
    'Security_1',
    'Security_2', 
    'Last_Spread',
    'Z_Score',
    'Max',
    'Min',
    'Last_vs_Max',
    'Last_vs_Min',
    'Percentile',
    'Custom_Sector_1',
    'Custom_Sector_2',
    'Yrs_Mat_Bucket_1',
    'Yrs_Mat_Bucket_2',
    'Size @ Best Offer_runs1',
    'Size @ Best Bid_runs2',
    'Bid/Offer_runs1',
    'Bid/Offer_runs2'
]].copy()

# Sort by Z_Score descending
g_z_own_executable_select = g_z_own_executable_select.sort_values(by='Z_Score', ascending=False)

# Identify float columns except the two size columns
size_cols = ['Size @ Best Offer_runs1', 'Size @ Best Bid_runs2']
float_cols = [col for col in g_z_own_executable_select.columns if g_z_own_executable_select[col].dtype == 'float' and col not in size_cols]

# Build the format dictionary
format_dict = {col: '{:.1f}' for col in float_cols}
format_dict.update({col: '{:,.0f}' for col in size_cols})

# Display with custom formatting
display(
    g_z_own_executable_select.head(50).style.format(format_dict)
)

,Security_1,Security_2,Last_Spread,Z_Score,Max,Min,Last_vs_Max,Last_vs_Min,Percentile,Custom_Sector_1,Custom_Sector_2,Yrs_Mat_Bucket_1,Yrs_Mat_Bucket_2,Size @ Best Offer_runs1,Size @ Best Bid_runs2,Bid/Offer_runs1,Bid/Offer_runs2
23299,CM 5.05 10/07/27,TD 5.491 09/08/28,-5.3,2.1,-4.7,-12.1,-0.5,6.9,99.6,Bail In,Bail In,2.1-3.1,3.1-4.1,"25,000,000","20,000,000",2.0,2.0
34208,CCDJ 4.407 05/19/27,TD 5.491 09/08/28,-9.5,1.9,-8.9,-16.2,-0.6,6.7,98.0,Non Major Senior Unsecured,Bail In,1-2.1,3.1-4.1,"5,000,000","20,000,000",2.0,2.0
48374,TD 4.21 06/01/27,TD 5.491 09/08/28,-8.4,1.7,-7.3,-16.3,-1.1,7.9,96.8,Bail In,Bail In,1-2.1,3.1-4.1,"10,000,000","20,000,000",2.0,2.0
75976,CNHI 4.8 03/25/27,TD 5.491 09/08/28,5.5,1.5,8.9,-7.7,-3.5,13.2,92.1,Industrials,Bail In,1-2.1,3.1-4.1,"4,000,000","20,000,000",2.0,2.0
128692,BMO 3.65 04/01/27,TD 5.491 09/08/28,-12.2,1.2,-6.5,-20.1,-5.7,7.9,91.3,Bail In,Bail In,1-2.1,3.1-4.1,"10,000,000","20,000,000",2.0,2.0
259524,COAGAS 4.691 09/30/29,TD 5.491 09/08/28,4.8,0.6,9.3,-6.9,-4.5,11.7,73.4,Pipeline,Bail In,4.1-5.1,3.1-4.1,"5,000,000","20,000,000",2.0,2.0
303675,LOWMAT 4.854 10/31/33,TD 5.491 09/08/28,12.9,0.4,19.9,4.3,-7.0,8.6,67.1,Utility,Bail In,7.1-10.1,3.1-4.1,"5,000,000","20,000,000",0.0,2.0
316398,GZMCN 4.67 09/27/32,TD 5.491 09/08/28,16.0,0.4,22.5,5.2,-6.6,10.8,58.7,Utility,Bail In,7.1-10.1,3.1-4.1,"3,000,000","20,000,000",1.0,2.0
340252,TD 3.6 10/31/2081,TD 5.491 09/08/28,863.3,0.3,1274.9,547.6,-411.6,315.7,63.5,Financial Hybrid,Bail In,>25.1,3.1-4.1,"5,000,000","20,000,000",-6.0,2.0
410194,RY 5.235 11/02/26,TD 5.491 09/08/28,-23.1,-0.1,-17.5,-29.1,-5.6,6.0,42.5,Bail In,Bail In,1-2.1,3.1-4.1,"10,000,000","20,000,000",2.0,2.0


## VS Decent Bid Offer + Size

In [21]:
# Filter for CAD currency, owned securities, tight bid/offer, and large offer size
g_z_bid_offer = g_z_all[
    (g_z_all['Currency_1'] == 'CAD') & 
    (g_z_all['Currency_2'] == 'CAD') &
    (g_z_all['Bid/Offer_runs1'] < 2) &
    (g_z_all['Size @ Best Offer_runs1'] > 3000000)
].copy()

# Select the same columns as before
g_z_bid_offer_select = g_z_bid_offer[[
    'Security_1',
    'Security_2', 
    'Last_Spread',
    'Z_Score',
    'Max',
    'Min',
    'Last_vs_Max',
    'Last_vs_Min',
    'Percentile',
    'Custom_Sector_1',
    'Custom_Sector_2',
    'Yrs_Mat_Bucket_1',
    'Yrs_Mat_Bucket_2',
    'Size @ Best Offer_runs1',
    'Size @ Best Bid_runs2',
    'Bid/Offer_runs1',
    'Bid/Offer_runs2'
]].copy()

# Sort by Z_Score descending
g_z_bid_offer_select = g_z_bid_offer_select.sort_values(by='Z_Score', ascending=False)

# Identify float columns except the two size columns
size_cols = ['Size @ Best Offer_runs1', 'Size @ Best Bid_runs2']
float_cols = [col for col in g_z_bid_offer_select.columns if g_z_bid_offer_select[col].dtype == 'float' and col not in size_cols]

# Build the format dictionary
format_dict = {col: '{:.1f}' for col in float_cols}
format_dict.update({col: '{:,.0f}' for col in size_cols})

# Display with custom formatting
display(
    g_z_bid_offer_select.head(30).style.format(format_dict)
)

,Security_1,Security_2,Last_Spread,Z_Score,Max,Min,Last_vs_Max,Last_vs_Min,Percentile,Custom_Sector_1,Custom_Sector_2,Yrs_Mat_Bucket_1,Yrs_Mat_Bucket_2,Size @ Best Offer_runs1,Size @ Best Bid_runs2,Bid/Offer_runs1,Bid/Offer_runs2
529,ALACN 2.477 11/30/30,NVACN 7 7/8 07/23/26,500.2,6.3,500.2,-431.2,0.0,931.4,100.0,Midstream,HY,5.1-7.1,1-2.1,"5,000,000","2,000,000",-2.0,nan
624,LOWMAT 4.854 10/31/33,NVACN 7 7/8 07/23/26,485.9,6.3,485.9,-443.7,0.0,929.5,100.0,Utility,HY,7.1-10.1,1-2.1,"5,000,000","2,000,000",0.0,nan
867,CM 1.96 04/21/31,NVACN 7 7/8 07/23/26,438.5,6.1,438.5,-488.4,0.0,926.9,100.0,NVCC Sub Debt,HY,5.1-7.1,1-2.1,"5,000,000","2,000,000",1.0,nan
900,CM 2.01 07/21/30,NVACN 7 7/8 07/23/26,409.9,5.8,409.9,-506.6,0.0,916.5,100.0,NVCC Sub Debt,HY,4.1-5.1,1-2.1,"5,000,000","2,000,000",1.0,nan
3024,ALACN 2.477 11/30/30,IPLCN 5.091 11/27/51,-104.0,2.9,-103.5,-129.8,-0.5,25.7,99.2,Midstream,Midstream,5.1-7.1,>25.1,"5,000,000","5,000,000",-2.0,nan
3920,ALACN 2.477 11/30/30,ALACN 8.9 11/10/2083,-130.4,2.8,-130.4,-229.0,0.0,98.6,100.0,Midstream,Non Financial Hybrids,5.1-7.1,>25.1,"5,000,000","2,000,000",-2.0,16.0
3968,ALACN 2.477 11/30/30,IFCCN 1.928 12/16/30,72.5,2.7,73.5,27.3,-1.0,45.2,98.0,Midstream,Insurance Senior,5.1-7.1,5.1-7.1,"5,000,000","2,000,000",-2.0,nan
4989,ALACN 2.477 11/30/30,GEICN 8.7 07/12/2083,-146.3,2.7,-146.3,-233.8,0.0,87.5,100.0,Midstream,Non Financial Hybrids,5.1-7.1,>25.1,"5,000,000","2,000,000",-2.0,17.0
5691,ALACN 2.477 11/30/30,HRUCN 2.633 02/19/27,14.0,2.6,14.0,-10.9,0.0,24.9,100.0,Midstream,REIT,5.1-7.1,1-2.1,"5,000,000","3,000,000",-2.0,nan
6999,ALACN 2.477 11/30/30,ALACN 7.35 08/17/2082,-107.2,2.5,-107.2,-226.8,0.0,119.5,100.0,Midstream,Non Financial Hybrids,5.1-7.1,>25.1,"5,000,000","2,000,000",-2.0,24.0


## CAD/USD RV

In [22]:
# Filter for CAD vs USD currency, same sector, same ticker, same maturity bucket
g_z_usd = g_z_all[
    (g_z_all['Currency_1'] == 'CAD') & 
    (g_z_all['Currency_2'] == 'USD') &
    (g_z_all['Custom_Sector_1'] == g_z_all['Custom_Sector_2']) &
    (g_z_all['Ticker_1'] == g_z_all['Ticker_2']) &
    (g_z_all['Yrs_Mat_Bucket_1'] == g_z_all['Yrs_Mat_Bucket_2'])
].copy()

# Select the same columns as before
g_z_usd_select = g_z_usd[[
    'Security_1',
    'Security_2', 
    'Last_Spread',
    'Z_Score',
    'Max',
    'Min',
    'Last_vs_Max',
    'Last_vs_Min',
    'Percentile',
    'XCCY',
    'Custom_Sector_1',
    'Custom_Sector_2',
    'Yrs_Mat_Bucket_1',
    'Yrs_Mat_Bucket_2',
    'Size @ Best Offer_runs1',
    'Size @ Best Bid_runs2',
    'Bid/Offer_runs1',
    'Bid/Offer_runs2'
]].copy()

# Sort by Z_Score descending
g_z_usd_select = g_z_usd_select.sort_values(by='Z_Score', ascending=False) #This would be your main toggle

# Identify float columns except the two size columns
size_cols = ['Size @ Best Offer_runs1', 'Size @ Best Bid_runs2']
float_cols = [col for col in g_z_usd_select.columns if g_z_usd_select[col].dtype == 'float' and col not in size_cols]

# Build the format dictionary
format_dict = {col: '{:.1f}' for col in float_cols}
format_dict.update({col: '{:,.0f}' for col in size_cols})

# Display with custom formatting
display(
    g_z_usd_select.head(30).style.format(format_dict))

,Security_1,Security_2,Last_Spread,Z_Score,Max,Min,Last_vs_Max,Last_vs_Min,Percentile,XCCY,Custom_Sector_1,Custom_Sector_2,Yrs_Mat_Bucket_1,Yrs_Mat_Bucket_2,Size @ Best Offer_runs1,Size @ Best Bid_runs2,Bid/Offer_runs1,Bid/Offer_runs2
109041,ENBCN 5 01/19/2082,ENBCN 7 3/8 01/15/2083,38.5,1.3,67.7,-94.0,-29.2,132.5,92.5,44.2,Non Financial Hybrids,Non Financial Hybrids,>25.1,>25.1,"2,000,000",nan,8.0,nan
122301,F 5.242 05/23/28,F 6.8 05/12/28,42.0,1.2,54.1,-54.8,-12.1,96.8,87.7,42.7,Auto,Auto,2.1-3.1,2.1-3.1,"3,000,000",nan,6.0,nan
126088,F 5.581 02/22/27,F 5.8 03/05/27,40.9,1.2,62.0,-37.3,-21.1,78.2,86.1,48.5,Auto,Auto,1-2.1,1-2.1,"5,000,000",nan,3.0,nan
169573,F 6.382 11/10/28,F 6.798 11/07/28,46.5,1.0,61.9,6.7,-15.4,39.8,81.0,43.4,Auto,Auto,3.1-4.1,3.1-4.1,"3,000,000",nan,9.0,nan
170551,TRPCN 4.2 03/04/2081,TRPCN 5.6 03/07/2082,27.9,1.0,57.5,-68.9,-29.6,96.8,86.5,21.6,Non Financial Hybrids,Non Financial Hybrids,>25.1,>25.1,"2,000,000",nan,5.0,nan
172514,F 5.581 02/22/27,F 5.85 05/17/27,30.9,0.9,44.5,-34.9,-13.5,65.8,84.9,35.1,Auto,Auto,1-2.1,1-2.1,"5,000,000",nan,3.0,nan
182838,BNS 3.7 07/27/2081,BNS 8 5/8 10/27/2082,985.3,0.9,1229.8,464.6,-244.5,520.7,77.8,997.5,Financial Hybrid,Financial Hybrid,>25.1,>25.1,"3,000,000",nan,44.0,nan
195344,BNS 3.7 07/27/2081,BNS 8 01/27/2084,948.0,0.8,1217.9,433.0,-269.9,515.0,76.6,955.3,Financial Hybrid,Financial Hybrid,>25.1,>25.1,"3,000,000",nan,44.0,nan
195700,MS 1.779 08/04/27,MS 5.05 01/28/27,9.8,0.8,19.7,-24.3,-9.9,34.1,77.8,38.2,Financial Maples,Financial Maples,1-2.1,1-2.1,"5,000,000",nan,10.0,nan
204336,BNS 7.023 07/27/2082,BNS 8 5/8 10/27/2082,42.4,0.8,78.0,-117.5,-35.6,159.9,82.5,42.1,Financial Hybrid,Financial Hybrid,>25.1,>25.1,"2,000,000",nan,42.0,nan


In [23]:
# Filter for CAD vs USD currency, same sector, same ticker, same maturity bucket
g_z_usd = g_z_all[
    (g_z_all['Currency_1'] == 'CAD') & 
    (g_z_all['Currency_2'] == 'USD') &
    (g_z_all['Custom_Sector_1'] == g_z_all['Custom_Sector_2']) &
    (g_z_all['Ticker_1'] == g_z_all['Ticker_2']) &
    (g_z_all['Yrs_Mat_Bucket_1'] == g_z_all['Yrs_Mat_Bucket_2'])
].copy()

# Select the same columns as before
g_z_usd_select = g_z_usd[[
    'Security_1',
    'Security_2', 
    'Last_Spread',
    'Z_Score',
    'Max',
    'Min',
    'Last_vs_Max',
    'Last_vs_Min',
    'Percentile',
    'XCCY',
    'Custom_Sector_1',
    'Custom_Sector_2',
    'Yrs_Mat_Bucket_1',
    'Yrs_Mat_Bucket_2',
    'Size @ Best Offer_runs1',
    'Size @ Best Bid_runs2',
    'Bid/Offer_runs1',
    'Bid/Offer_runs2'
]].copy()

# Sort by Z_Score descending
g_z_usd_select = g_z_usd_select.sort_values(by='Z_Score', ascending=True) #This would be your main toggle

# Identify float columns except the two size columns
size_cols = ['Size @ Best Offer_runs1', 'Size @ Best Bid_runs2']
float_cols = [col for col in g_z_usd_select.columns if g_z_usd_select[col].dtype == 'float' and col not in size_cols]

# Build the format dictionary
format_dict = {col: '{:.1f}' for col in float_cols}
format_dict.update({col: '{:,.0f}' for col in size_cols})

# Display with custom formatting
display(
    g_z_usd_select.head(30).style.format(format_dict))

,Security_1,Security_2,Last_Spread,Z_Score,Max,Min,Last_vs_Max,Last_vs_Min,Percentile,XCCY,Custom_Sector_1,Custom_Sector_2,Yrs_Mat_Bucket_1,Yrs_Mat_Bucket_2,Size @ Best Offer_runs1,Size @ Best Bid_runs2,Bid/Offer_runs1,Bid/Offer_runs2
7883,DE 1.34 09/08/27,DE 4 3/4 01/20/28,-14.1,-2.5,16.3,-23.0,-30.4,9.0,1.6,-14.5,Industrials,Industrials,2.1-3.1,2.1-3.1,nan,nan,nan,nan
8831,HNDA 1.646 02/25/28,HNDA 2 03/24/28,-38.2,-2.4,20.2,-38.6,-58.4,0.4,0.8,-34.9,Auto,Auto,2.1-3.1,2.1-3.1,"3,000,000",nan,9.0,nan
14306,TD 1.888 03/08/28,TD 5.523 07/17/28,-34.3,-2.3,-1.7,-34.6,-32.6,0.3,0.8,-34.5,Bail In,Bail In,2.1-3.1,2.1-3.1,"2,000,000",nan,10.0,nan
18110,HNDA 1.646 02/25/28,HNDA 4.7 01/12/28,-32.2,-2.2,29.6,-32.6,-61.8,0.4,1.6,-31.7,Auto,Auto,2.1-3.1,2.1-3.1,"3,000,000",nan,9.0,nan
19340,HNDA 1.711 09/28/26,HNDA 4.9 03/12/27,-34.1,-2.1,18.4,-35.9,-52.5,1.8,0.8,-28.1,Auto,Auto,1-2.1,1-2.1,"5,000,000",nan,5.0,nan
20027,HNDA 1.711 09/28/26,HNDA 4.9 07/09/27,-36.0,-2.1,14.4,-36.0,-50.4,0.0,0.4,-31.7,Auto,Auto,1-2.1,1-2.1,"5,000,000",nan,5.0,nan
21664,TD 1.888 03/08/28,TD 5.156 01/10/28,-32.9,-2.1,4.1,-32.9,-37.0,0.0,0.4,-32.2,Bail In,Bail In,2.1-3.1,2.1-3.1,"2,000,000",nan,10.0,nan
22968,DE 1.34 09/08/27,DE 4.15 09/15/27,-13.3,-2.1,16.9,-14.6,-30.2,1.3,1.2,-11.3,Industrials,Industrials,2.1-3.1,2.1-3.1,nan,nan,nan,nan
23484,TOYOTA 4.33 01/24/28,TOYOTA 4.55 09/20/27,12.6,-2.1,42.9,10.3,-30.3,2.3,1.2,14.7,Auto,Auto,2.1-3.1,2.1-3.1,"5,000,000",nan,5.0,nan
26014,HNDA 1.711 09/28/26,HNDA 2.534 03/10/27,-32.7,-2.0,22.6,-36.4,-55.3,3.7,1.6,-26.2,Auto,Auto,1-2.1,1-2.1,"5,000,000",nan,5.0,nan
